In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
import torchsummary
from torchsummary import summary
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from model import LeNet5

In [ ]:
model = LeNet5()

In [ ]:
summary(model,(1,28,28))

In [ ]:
file = './RBF_kernel/0_RBF.jpg'
image = cv2.imread(file, 0)
image = cv2.threshold(image,127,1,cv2.THRESH_BINARY)[1]*-1+1
plt.imshow(image)

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize((0.05,), (0.5,))
    ])

train_data = datasets.MNIST(root='data/', train=True, transform=transform, download=True)
test_data = datasets.MNIST(root='data/', train=False, transform=transform, download=True)

# data loader
train_loader = DataLoader(dataset = train_data, batch_size=256, shuffle=False)
test_loader = DataLoader(dataset = test_data, batch_size=256, shuffle=False)


In [ ]:
def train(model, epoch=100, w_decay=0):
    # optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=w_decay)
    
    # loss function
    def loss_fn(output, target):
        loss = output[target==1].pow(2).sum()
        loss += torch.log(np.exp(-0.1)+torch.exp(-output[target==0]).sum())
        return loss

    # training
    train_loss_list=[]
    train_accuracy_list=[]
    for i in range(epoch):
        train_loss = 0
        train_accuracy = 0
        #load data
        for data, target in train_loader:
            data = torch.Tensor(np.pad(data,((0,0),(0,0),(2,2),(2,2)),'constant', constant_values=-0.1))
            target = nn.functional.one_hot(target).float()
            optimizer.zero_grad()

            #forward propagation
            y_pred = model(data)
            loss = loss_fn(y_pred,target)
            
           #backpropagation
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_accuracy += (torch.argmin(model(data),dim=1)==torch.argmax(target,dim=1)).sum().item()


        train_loss /= len(train_loader.dataset)
        train_loss *= 256
        train_loss_list.append(train_loss)

        train_accuracy /= len(train_loader.dataset)
        train_accuracy_list.append(train_accuracy)


        if i%1==0:
            print('epoch : {}/{}, train_loss : {:.8f}, train_acc : {:.2f}'.format(i+1,epoch,train_loss,train_accuracy))
    return(train_loss_list,train_accuracy_list)

In [ ]:
def test(model):
    accuracy = 0
    for data, target in test_loader:
        data = torch.Tensor(np.pad(data,((0,0),(0,0),(2,2),(2,2)),'constant', constant_values=-0.1))
        y_pred = model(data)
        accuracy += (torch.argmin(y_pred,dim=1)==target).sum().item()
    accuracy /= len(test_loader.dataset)
    return(accuracy)

In [ ]:
train_hist_loss, train_hist_acc = train(model,20)

In [ ]:
torch.save(model,'LeNet_1998.pt')
test(model)